In [55]:
import pandas as pd
import numpy as np
import requests
import time

In [56]:
brent_raw = pd.read_csv("../data/brent.csv")

# Оставляем только дату и цену
brent = brent_raw[["Дата", "Цена"]].copy()

# Чистим цену — убираем запятые, переводим в float
brent["Цена"] = brent["Цена"].str.replace(",", ".").astype(float)

# Дата
brent["Дата"] = pd.to_datetime(brent["Дата"], format="%d.%m.%Y")
brent = brent.rename(columns={"Дата": "Date", "Цена": "brent_close"})
brent["Date"] = brent["Date"].dt.date

# Сортируем по дате
brent = brent.sort_values("Date").reset_index(drop=True)

print(f"Brent: {len(brent)} строк")
print(brent.head())

Brent: 2582 строк
         Date  brent_close
0  2016-01-04        37.22
1  2016-01-05        36.42
2  2016-01-06        34.23
3  2016-01-07        33.75
4  2016-01-08        33.55


In [57]:
def load_moex_currency(start="2016-01-01", end="2026-01-01", interval=24):
    all_candles = []
    start_index = 0

    while True:
        url = ("https://iss.moex.com/iss/engines/currency/markets/selt/"
               "boards/CETS/securities/USD000UTSTOM/candles.json")
        params = {"from": start, "till": end, "interval": interval, "start": start_index}
        response = requests.get(url, params=params)
        data = response.json()
        candles = data["candles"]["data"]

        if not candles:
            break

        all_candles.extend(candles)
        start_index += len(candles)

        if len(candles) < 500:
            break

        time.sleep(0.3)

    df = pd.DataFrame(all_candles, columns=data["candles"]["columns"])
    return df

print("Загружаю USD/RUB...")
usdrub_raw = load_moex_currency()
usdrub = usdrub_raw[["begin", "close"]].rename(columns={"begin": "Date", "close": "usdrub_close"})
usdrub["Date"] = pd.to_datetime(usdrub["Date"]).dt.date
usdrub = usdrub.sort_values("Date").reset_index(drop=True)
print(f"USD/RUB: {len(usdrub)} строк")

Загружаю USD/RUB...
USD/RUB: 2133 строк


In [58]:
def load_moex_index(ticker, start="2016-01-01", end="2026-01-01", interval=24):
    all_candles = []
    start_index = 0

    while True:
        url = (f"https://iss.moex.com/iss/engines/stock/markets/index/"
               f"boards/SNDX/securities/{ticker}/candles.json")
        params = {"from": start, "till": end, "interval": interval, "start": start_index}
        response = requests.get(url, params=params)
        data = response.json()
        candles = data["candles"]["data"]

        if not candles:
            break

        all_candles.extend(candles)
        start_index += len(candles)

        if len(candles) < 500:
            break

        time.sleep(0.3)

    df = pd.DataFrame(all_candles, columns=data["candles"]["columns"])
    return df

print("Загружаю IMOEX...")
imoex_raw = load_moex_index("IMOEX")
imoex = imoex_raw[["begin", "close"]].rename(columns={"begin": "Date", "close": "imoex_close"})
imoex["Date"] = pd.to_datetime(imoex["Date"]).dt.date
imoex = imoex.sort_values("Date").reset_index(drop=True)
print(f"IMOEX: {len(imoex)} строк")

Загружаю IMOEX...
IMOEX: 2514 строк


In [59]:
def add_macro_features(df, col, prefix):
    df = df.copy().sort_values("Date")

    df[f"{prefix}_return_1d"]  = df[col].pct_change(1)
    df[f"{prefix}_return_5d"]  = df[col].pct_change(5)
    df[f"{prefix}_return_20d"] = df[col].pct_change(20)
    df[f"{prefix}_to_ma20"]    = df[col] / df[col].rolling(20).mean() - 1
    df[f"{prefix}_vol_10d"]    = df[f"{prefix}_return_1d"].rolling(10).std()

    return df.drop(columns=[col])

brent  = add_macro_features(brent,  "brent_close",  "brent")
usdrub = add_macro_features(usdrub, "usdrub_close", "usdrub")
imoex  = add_macro_features(imoex,  "imoex_close",  "imoex")

print("Макропризнаки посчитаны!")
print(f"Brent:  {brent.columns.tolist()}")
print(f"USDRUB: {usdrub.columns.tolist()}")
print(f"IMOEX:  {imoex.columns.tolist()}")

Макропризнаки посчитаны!
Brent:  ['Date', 'brent_return_1d', 'brent_return_5d', 'brent_return_20d', 'brent_to_ma20', 'brent_vol_10d']
USDRUB: ['Date', 'usdrub_return_1d', 'usdrub_return_5d', 'usdrub_return_20d', 'usdrub_to_ma20', 'usdrub_vol_10d']
IMOEX:  ['Date', 'imoex_return_1d', 'imoex_return_5d', 'imoex_return_20d', 'imoex_to_ma20', 'imoex_vol_10d']


In [60]:
df = pd.read_csv("../data/data_features.csv", parse_dates=["Date"])
df["Date"] = pd.to_datetime(df["Date"]).dt.date

df = df.merge(brent,  on="Date", how="left")
df = df.merge(usdrub, on="Date", how="left")
df = df.merge(imoex,  on="Date", how="left")

# Заполняем пропуски по выходным
macro_cols = [c for c in df.columns if c.startswith(("brent_", "usdrub_", "imoex_"))]
df[macro_cols] = df[macro_cols].ffill()

df = df.dropna().reset_index(drop=True)
df["Date"] = pd.to_datetime(df["Date"])

print(f"Строк: {len(df)}, Колонок: {len(df.columns)}")
print(f"Макропризнаков добавлено: {len(macro_cols)}")

Строк: 37603, Колонок: 57
Макропризнаков добавлено: 15


In [61]:
df.to_csv("../data/data_macro.csv", index=False)
print("Сохранено в data/data_macro.csv!")

Сохранено в data/data_macro.csv!
